# Chapter 04 제출 답안 양식. pandas로 데이터에 질문하기

> 주 제출물은 실행 완료 Notebook `chapter04/chapter04.ipynb`입니다. 이 양식의 항목을 Notebook의 Markdown 셀로 추가해 작성합니다.

## 0. 제출 정보
- 이름: 윤강
- GitHub ID: yg920
- 작성일: 2023. 9. 10.
- 최종 제출 URL: 

## 1. 질문과 필요한 데이터 선택

### 내가 확인하려는 질문
• 이 코드는 어떤 분석 질문에 답하는가?

• 어떤 DataFrame과 컬럼이 필요한가?

• 결과 표에서 한 행은 무엇을 의미하는가?

• 전체 주문과 완료 주문 중 어느 범위를 사용하는가?

• 행 수와 실제 주문 수는 어떻게 다른가?

• 병합 전후에 무엇을 검증해야 하는가?

### 사용한 파일/컬럼
orders.csv
 - order_id
 - customer_id
 - order_date
 - order_status 

order_items.csv
 - order_id
 - product_id
 - quantity
 - unit_price

products.csv
 - product_id
 - product_name
 - category 

customers.csv
 - customer_id
 - name
 - city

### 결과 관찰
orders에는 주문 단위의 정보가 있고, order_items에는 주문에 포함된 상품별 수량과 판매 단가가 있다. 
products에는 상품명과 카테고리가 있으며, customers에는 주문 고객의 정보가 있다. 
따라서 주문, 주문상세, 상품, 고객 데이터를 서로 연결하면 완료된 주문의 매출을 카테고리, 상품, 월, 고객별로 분석할 수 있다.

### 나의 해석과 판단
출을 확인하기 위해서는 주문 상태와 주문 날짜가 필요하고, 실제 판매 금액을 계산하기 위해 quantity와 unit_price가 필요하다. 
또한 어떤 상품과 카테고리에서 매출이 발생했는지 확인하기 위해 product_id, product_name, category를 선택했다. 
고객별 분석을 위해 customer_id와 name도 함께 사용하기로 했다.

### 업무·분석적 의미
이 데이터를 분석하면 어떤 상품이나 카테고리가 매출에 많이 기여하는지 파악할 수 있다. 
또한 월별 매출 변화와 주요 고객을 확인하여 상품 운영이나 판매 전략을 세울 때 활용할 수 있다.

### 한계와 추가 확인 사항
현재 데이터만으로는 할인, 쿠폰, 배송비, 환불 금액 등의 정보를 확인할 수 없다.
따라서 계산한 판매 금액을 실제 회계상 순매출과 동일하다고 단정할 수 없다.

In [8]:
import pandas as pd
from pathlib import Path

current = Path.cwd().resolve()

while True:
    if (current / "data" / "raw").exists():
        PROJECT_ROOT = current
        break

    if current.parent == current:
        raise FileNotFoundError("data/raw 폴더를 찾을 수 없습니다.")

    current = current.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"

print("프로젝트 루트:", PROJECT_ROOT)
print("데이터 폴더:", DATA_DIR)
print("데이터 폴더 존재 여부:", DATA_DIR.exists())

order_items = pd.read_csv(DATA_DIR / "order_items.csv")

order_items.head()

프로젝트 루트: C:\dev\llm_data_analysis-course
데이터 폴더: C:\dev\llm_data_analysis-course\data\raw
데이터 폴더 존재 여부: True


,order_item_id,order_id,product_id,quantity,unit_price
0,1,1,100,3,102000
1,2,1,87,5,25000
2,3,1,7,3,142000
3,4,1,9,3,193000
4,5,2,72,4,189000


In [9]:
# quantity가 2개 이상인 주문 항목만 선택
filtered_items = order_items[
    order_items["quantity"] >= 2
].copy()

# 주문 항목별 판매 금액 계산
filtered_items["line_total"] = (
    filtered_items["quantity"]
    * filtered_items["unit_price"]
)

# 판매 금액이 큰 순서대로 정렬
filtered_items = filtered_items.sort_values(
    by="line_total",
    ascending=False
)

print(filtered_items.head())

     order_item_id  order_id  product_id  quantity  unit_price  line_total
22              23        10          99         5      200000     1000000
85              86        39          99         5      200000     1000000
183            184        76          99         5      200000     1000000
78              79        35          70         5      198000      990000
493            494       190          70         5      198000      990000


## 2. 필터링·정렬·파생 컬럼
- 적용한 필터 조건: quantity >= 2
- 정렬 기준: line_total 기준 내림차순
- 만든 파생 컬럼: line_total
- `line_total` 계산식: 
filtered_items = order_items[order_items["quantity"] >= 2].copy()

filtered_items["line_total"] = (
filtered_items["quantity"] * filtered_items["unit_price"]
)

filtered_items = filtered_items.sort_values( by="line_total", ascending=False ) filtered_items.head()

![필터와 파생 컬럼](images/step02_transform.png)

### 결과 관찰

### 나의 해석과 판단
quantity를 기준으로 필터링하면 여러 개 판매된 상품을 중심으로 확인할 수 있다. 
만약 quantity 기준을 3개 이상이나 4개 이상으로 높이면 조회되는 데이터 수는 줄어들고 대량 구매에 가까운 주문만 남게 된다. 
반대로 필터 조건을 낮추거나 제거하면 더 많은 주문 항목이 포함되므로 분석 결과의 범위도 달라질 수 있다.

### 업무·분석적 의미
line_total을 이용하면 단순히 판매 수량만 보는 것이 아니라 각 주문 항목이 실제 판매 금액에 얼마나 기여했는지 확인할 수 있다. 
판매 금액이 큰 주문 항목을 확인하면 어떤 상품이 매출에 크게 기여하는지 추가로 분석하는 데 활용할 수 있다.

### 한계와 추가 확인 사항
현재 line_total은 quantity와 unit_price만 이용해 계산한 값이다. 
할인, 쿠폰, 배송비, 환불 등의 정보가 반영되지 않았기 때문에 이 값을 실제 최종 결제 금액이나 순매출이라고 단정할 수 없다. 
또한 주문 상태를 아직 적용하지 않았으므로 cancelled나 refunded 주문이 포함될 수 있는지도 다음 단계에서 확인해야 한다.

In [10]:
orders = pd.read_csv(DATA_DIR / "orders.csv")
products = pd.read_csv(DATA_DIR / "products.csv")
customers = pd.read_csv(DATA_DIR / "customers.csv")

print("orders:", orders.shape)
print("order_items:", order_items.shape)
print("products:", products.shape)
print("customers:", customers.shape)

orders: (300, 5)
order_items: (764, 5)
products: (100, 4)
customers: (150, 6)


In [11]:
# 병합 전 order_items 행 수 확인
before_rows = len(order_items)

# order_items + orders
merged = order_items.merge(
    orders[["order_id", "customer_id", "order_date", "order_status"]],
    on="order_id",
    how="left",
    validate="many_to_one"
)

# + products
merged = merged.merge(
    products[["product_id", "product_name", "category"]],
    on="product_id",
    how="left",
    validate="many_to_one"
)

# + customers
merged = merged.merge(
    customers[["customer_id", "name", "city"]],
    on="customer_id",
    how="left",
    validate="many_to_one"
)

# 병합 후 행 수 확인
after_rows = len(merged)

print("병합 전 행 수:", before_rows)
print("병합 후 행 수:", after_rows)

merged.head()

병합 전 행 수: 764
병합 후 행 수: 764


,order_item_id,order_id,product_id,quantity,unit_price,customer_id,order_date,order_status,product_name,category,name,city
0,1,1,100,3,102000,123,2026-07-02,completed,도서 상품 100,도서,김재현,서울
1,2,1,87,5,25000,123,2026-07-02,completed,도서 상품 087,도서,김재현,서울
2,3,1,7,3,142000,123,2026-07-02,completed,도서 상품 007,도서,김재현,서울
3,4,1,9,3,193000,123,2026-07-02,completed,스포츠 상품 009,스포츠,김재현,서울
4,5,2,72,4,189000,77,2025-09-17,cancelled,뷰티 상품 072,뷰티,김은주,서울


In [12]:
print("order_id 누락:", merged["order_id"].isna().sum())
print("product_name 누락:", merged["product_name"].isna().sum())
print("customer_id 누락:", merged["customer_id"].isna().sum())
print("name 누락:", merged["name"].isna().sum())

order_id 누락: 0
product_name 누락: 0
customer_id 누락: 0
name 누락: 0


## 3. merge 검증
- 병합한 데이터: order_items + orders + products + customers
- 사용한 key: order_id, product_id, customer_id
- validate 결과: many_to_one 조건으로 정상 병합
- indicator 결과: 매칭 여부를 확인하여 비정상 병합이 없는지 점검
- 병합 전/후 행 수: order_items의 행 수를 기준으로 병합 후에도 행 수가 유지되는지 확인

![merge 검증](images/step03_merge.png)


### 결과 관찰
order_items를 기준으로 orders, products, customers 데이터를 순서대로 병합했다.
병합 후 확인한 결과 order_id, product_name, customer_id, name 컬럼의 결측값이 모두 0건으로 나타났다.
따라서 각 주문상세 데이터가 주문, 상품, 고객 데이터와 정상적으로 연결된 것을 확인했다.

### 나의 해석과 판단
이번 병합은 각 단계에서 validate="many_to_one" 조건을 사용하여 관계가 예상한 구조와 맞는지 확인했다.
또한 병합 후 주요 컬럼의 결측값이 모두 0건이므로 연결되지 않은 주문, 상품, 고객 데이터는 없는 것으로 판단했다.
병합 전후의 행 수도 동일하다면 병합 과정에서 데이터가 불필요하게 증가하거나 감소하지 않았다고 볼 수 있다.

### 업무·분석적 의미
merge가 잘못되면 하나의 주문 항목이 여러 행으로 중복되어 매출이 실제보다 크게 계산될 수 있고, 반대로 일부 데이터가 매칭되지 않으면 매출이 누락될 수 있다.
따라서 집계 분석을 하기 전에 병합 관계와 행 수, 결측 여부를 검증하는 과정이 필요하다.

### 한계와 추가 확인 사항
주요 컬럼에서 결측값이 없더라도 데이터 값 자체가 올바른지는 별도로 확인할 필요가 있다.
또한 병합 전후 행 수가 동일한지 확인하여 데이터 중복이나 누락이 발생하지 않았는지도 함께 검증해야 한다.

In [13]:
# 완료 주문만 선택
completed = merged[
    merged["order_status"] == "completed"
].copy()

# 주문 항목별 판매 금액 계산
completed["line_total"] = (
    completed["quantity"]
    * completed["unit_price"]
)

# 주문 날짜를 datetime으로 변환
completed["order_date"] = pd.to_datetime(
    completed["order_date"]
)

# 월 단위 분석용 컬럼 생성
completed["order_month"] = (
    completed["order_date"]
    .dt.to_period("M")
    .astype(str)
)

print("전체 주문상세 행 수:", len(merged))
print("completed 주문상세 행 수:", len(completed))

completed.head()

전체 주문상세 행 수: 764
completed 주문상세 행 수: 474


,order_item_id,order_id,product_id,quantity,unit_price,customer_id,order_date,order_status,product_name,category,name,city,line_total,order_month
0,1,1,100,3,102000,123,2026-07-02,completed,도서 상품 100,도서,김재현,서울,306000,2026-07
1,2,1,87,5,25000,123,2026-07-02,completed,도서 상품 087,도서,김재현,서울,125000,2026-07
2,3,1,7,3,142000,123,2026-07-02,completed,도서 상품 007,도서,김재현,서울,426000,2026-07
3,4,1,9,3,193000,123,2026-07-02,completed,스포츠 상품 009,스포츠,김재현,서울,579000,2026-07
12,13,6,83,3,24000,87,2026-05-16,completed,전자기기 상품 083,전자기기,유서현,울산,72000,2026-05


In [14]:
# 카테고리별 매출
category_sales = (
    completed.groupby("category", as_index=False)
    ["line_total"]
    .sum()
    .rename(columns={"line_total": "total_sales"})
    .sort_values("total_sales", ascending=False)
)

# 상품별 매출
product_sales = (
    completed.groupby(
        ["product_id", "product_name"],
        as_index=False
    )["line_total"]
    .sum()
    .rename(columns={"line_total": "total_sales"})
    .sort_values("total_sales", ascending=False)
)

# 월별 매출
monthly_sales = (
    completed.groupby("order_month", as_index=False)
    ["line_total"]
    .sum()
    .rename(columns={"line_total": "total_sales"})
    .sort_values("order_month")
)

# 고객별 매출
customer_sales = (
    completed.groupby(
        ["customer_id", "name"],
        as_index=False
    )["line_total"]
    .sum()
    .rename(columns={"line_total": "total_sales"})
    .sort_values("total_sales", ascending=False)
)

In [15]:
print("=== 카테고리별 매출 ===")
display(category_sales.head())

print("=== 상품별 매출 ===")
display(product_sales.head())

print("=== 월별 매출 ===")
display(monthly_sales)

print("=== 고객별 매출 ===")
display(customer_sales.head())

=== 카테고리별 매출 ===


,category,total_sales
3,스포츠,31743000
5,전자기기,26400000
2,생활용품,23915000
1,뷰티,23383000
4,식품,16573000


=== 상품별 매출 ===


,product_id,product_name,total_sales
39,41,스포츠 상품 041,5705000
11,12,식품 상품 012,4375000
8,9,스포츠 상품 009,3860000
70,72,뷰티 상품 072,3780000
69,71,전자기기 상품 071,3703000


=== 월별 매출 ===


,order_month,total_sales
0,2025-09,7190000
1,2025-10,16291000
2,2025-11,13704000
3,2025-12,23360000
4,2026-01,7282000
5,2026-02,10851000
6,2026-03,17538000
7,2026-04,9589000
8,2026-05,14798000
9,2026-06,15402000


=== 고객별 매출 ===


,customer_id,name,total_sales
76,117,김지원,4100000
62,102,이진호,3996000
51,83,전은경,3880000
21,30,이민재,3590000
29,40,박예준,3523000


## 4. completed 주문 범위와 집계
- 분석 범위 정의: order_status가 completed인 주문만 분석 대상으로 사용
- 카테고리별 결과: 스포츠 카테고리가 가장 높은 매출을 기록함
- 상품별 결과: 스포츠상품 041이 가장 높은 매출을 기록함
- 월별 결과: 2025-09의 매출이 가장 높게 나타남
- 고객별 결과: 김지원 고객의 매출이 가장 높게 나타남

![핵심 집계 결과](images/step04_groupby.png)

### 결과 관찰
completed 상태의 주문만 대상으로 매출을 집계한 결과, 카테고리별 매출에서는 스포츠가 가장 높은 것으로 나타났다.
상품별 매출에서는 스포츠상품 041이 가장 높은 매출을 기록했다.
월별 매출에서는 2025-09가 가장 높은 매출이 발생한 월이었고, 고객별 매출에서는 김지원 고객의 매출 기여도가 가장 높았다.

### 나의 해석과 판단
가장 의미 있다고 본 결과는 스포츠 카테고리와 스포츠상품 041의 매출 기여도가 높았다는 점이다.
카테고리와 상품 단위 결과가 같은 방향을 보이기 때문에 스포츠 관련 상품이 전체 매출에서 중요한 비중을 차지하고 있을 가능성이 있다고 판단했다.
또한 2025-09에 매출이 가장 높았기 때문에 해당 월에 특정 프로모션, 계절적 요인, 주문 증가 등이 있었는지 추가로 확인할 필요가 있다.

### 업무·분석적 의미
스포츠 카테고리와 스포츠상품 041의 판매 원인을 추가로 분석하면 인기 상품 운영이나 재고 관리, 프로모션 전략에 활용할 수 있다.
2025-09의 매출 증가 원인을 파악하면 향후 비슷한 시기에 어떤 판매 전략을 적용할지 판단하는 데 도움이 될 수 있다.
고객별로는 김지원 고객의 구매 내역을 확인하여 구매 빈도와 상품 구성을 분석하면 주요 고객의 구매 패턴을 파악하는 데 활용할 수 있다.

### 한계와 추가 확인 사항
현재 total_sales는 quantity와 unit_price를 곱한 line_total의 합계이므로 실제 회계상 순매출과 같다고 단정할 수 없다.
할인, 쿠폰, 배송비, 환불 금액, 세금 등의 정보가 포함되어 있지 않기 때문이다.
또한 특정 카테고리나 고객의 매출이 높은 이유를 정확히 판단하려면 주문 건수, 구매 수량, 평균 주문금액 등의 추가 지표도 함께 확인할 필요가 있다.


In [16]:
# completed 주문의 원본 총합
original_total = completed["line_total"].sum()

# 각 집계 결과의 총합
category_total = category_sales["total_sales"].sum()
monthly_total = monthly_sales["total_sales"].sum()
customer_total = customer_sales["total_sales"].sum()

print("원본 completed line_total 합계:", original_total)
print("카테고리 합계:", category_total)
print("월별 합계:", monthly_total)
print("고객별 합계:", customer_total)

print("카테고리 차이:", original_total - category_total)
print("월별 차이:", original_total - monthly_total)
print("고객별 차이:", original_total - customer_total)

원본 completed line_total 합계: 148990000
카테고리 합계: 148990000
월별 합계: 148990000
고객별 합계: 148990000
카테고리 차이: 0
월별 차이: 0
고객별 차이: 0


## 5. 총합 일치 검증 
- 원본 completed `line_total` 합계: 148990000
- 카테고리 합계: 148990000
- 월별 합계: 148990000
- 고객별 합계: 148990000
- 차이 여부: 0(일치)
 
![총합 검증](images/step05_total_check.png)

### 나의 해석과 판단

총합 검증은 groupby 과정에서 데이터가 누락되거나 중복 집계되지 않았는지 확인하기 위해 필요하다.
원본 completed 데이터의 line_total 합계와 카테고리별, 월별, 고객별 집계 결과의 합계가 동일하다면 집계 과정이 일관되게 이루어졌다고 판단할 수 있다.
만약 총합이 일치하지 않는다면 먼저 분석 범위가 동일한지 확인하고, 그다음 결측값으로 인해 groupby에서 제외된 데이터가 있는지 확인해야 한다. 이후 merge 과정에서 행이 증가하거나 감소했는지, 중복된 데이터가 있는지도 순서대로 점검해야 한다.


## 6. LLM pandas 코드 검증
- LLM Prompt 요약: 완료된 주문만 대상으로 상품별 매출을 계산하고, 매출이 높은 상품부터 정렬하는 pandas 코드를 작성해줘.
- 제안 코드 요약: result = (
    merged[merged["status"] == "completed"]
    .groupby("product_name")["sales"]
    .sum()
    .sort_values(ascending=False)
)

result.head()

- 실제 컬럼/범위와 맞지 않은 부분: status      → order_status   sales       → 존재하지 않음
- 수정한 내용: 실제 판매금액은 quantity * unit_price로 계산해야 한다
- 최종 판단: 수정 후 사용

![LLM 코드 검증](images/step06_llm_validation.png)

### 나의 해석과 판단
LLM이 작성한 코드가 오류 없이 실행된다고 해서 분석 결과까지 정확하다고 볼 수는 없다.
코드가 실제 데이터의 컬럼명과 분석 범위를 올바르게 사용했는지 확인해야 하며, completed 주문만 분석해야 하는데 전체 주문이 포함되거나 잘못된 컬럼으로 금액을 계산하면 실행은 되더라도 분석 결과가 달라질 수 있다.
따라서 LLM이 제안한 코드는 실제 데이터 구조와 분석 목적을 기준으로 검증하고, 기존에 계산한 결과와 비교한 뒤 사용하는 것이 필요하다고 판단했다.

In [18]:
llm_check = merged[
    merged["order_status"] == "completed"
].copy()

llm_check["line_total"] = (
    llm_check["quantity"]
    * llm_check["unit_price"]
)

llm_product_sales = (
    llm_check
    .groupby("product_name", as_index=False)["line_total"]
    .sum()
    .rename(columns={"line_total": "total_sales"})
    .sort_values("total_sales", ascending=False)
)

llm_product_sales.head()


print("기존 상품 1위:")
display(product_sales.head(1))

print("LLM 코드 수정 후 상품 1위:")
display(llm_product_sales.head(1))

기존 상품 1위:


,product_id,product_name,total_sales
39,41,스포츠 상품 041,5705000


LLM 코드 수정 후 상품 1위:


,product_name,total_sales
54,스포츠 상품 041,5705000


## 7. Chapter 04 최종 인사이트
### 가장 의미 있다고 생각한 결과 2가지
1. 스포츠 카테고리가 가장 높은 매출을 기록했고, 상품별로는 스포츠상품 041의 매출이 가장 높게 나타났다.
2. 월별 매출에서는 2025-09가 가장 높았고, 고객별 매출에서는 김지원 고객의 매출 기여도가 가장 높게 나타났다.

### 그 결과를 뒷받침하는 수치/표
completed 주문만 대상으로 집계했을 때 전체 line_total 합계는 148,990,000이었다.
카테고리별, 월별, 고객별 집계 결과의 총합도 모두 148,990,000으로 동일했고, 각 집계와 원본 합계의 차이는 0으로 확인되었다.
이를 통해 이번 집계 과정에서 매출 데이터의 누락이나 중복 없이 일관되게 분석되었음을 확인했다.

### 추가로 확인하고 싶은 질문
스포츠 카테고리와 스포츠상품 041의 매출이 높은 이유가 실제 판매 수량 때문인지, 상품 단가가 높기 때문인지 추가로 확인하고 싶다.
또한 2025-09의 매출이 다른 월보다 높은 원인이 특정 프로모션이나 계절적 요인 때문인지 확인하고 싶다.
김지원 고객의 높은 매출이 여러 번 반복 구매한 결과인지, 한 번의 고액 주문 때문인지도 추가로 분석할 필요가 있다.

### 현재 결과의 한계
현재 total_sales는 quantity와 unit_price를 곱한 line_total의 합계로 계산했다.
따라서 할인, 쿠폰, 배송비, 세금, 환불 금액 등이 반영되지 않았기 때문에 실제 회계상 순매출과 동일하다고 단정할 수 없다.
또한 현재 분석은 매출 금액을 중심으로 진행했기 때문에 주문 건수, 평균 주문금액, 구매 빈도, 상품별 판매 수량 등의 지표도 함께 확인해야 더 정확한 분석이 가능하다.

## 최종 제출 체크
- [o] 핵심 셀 Output이 남아 있습니다.
- [o] merge와 총합 검증 Evidence가 있습니다.
- [o] 결과 관찰과 해석이 구분되어 있습니다.
- [o] LLM 코드를 검증했습니다.
- [o] 개인정보/Secret이 없습니다.
- [o] `chapter04/chapter04.ipynb`가 GitHub에서 정상 표시됩니다.
- [o] 최종 Notebook 파일 URL을 제출합니다.